# Indian Student Name Extraction Tool (Colab + Localhost Jupyter)
Upload one or more images, extract text using EasyOCR, detect Indian student names, parse names, and export results to Excel.


In [ ]:
# 1) Install dependencies (run this cell in Colab/Notebook)
!pip -q install easyocr openpyxl ipywidgets


In [ ]:
# 2) Imports
import re
from pathlib import Path
from typing import List, Dict

import cv2
import easyocr
import pandas as pd
from IPython.display import display

try:
    from google.colab import files as colab_files
    IS_COLAB = True
except Exception:
    colab_files = None
    IS_COLAB = False


In [ ]:
# 3) Embedded Indian first-name lists for gender guess (case-insensitive)
male_names = {
    'aarav', 'aditya', 'akash', 'aman', 'amit', 'anil', 'arjun', 'arvind', 'ashok',
    'ayush', 'bhavesh', 'deepak', 'dinesh', 'gaurav', 'harish', 'himanshu', 'karthik',
    'krishna', 'manish', 'mohit', 'mukesh', 'naveen', 'nikhil', 'pradeep', 'rahul',
    'raj', 'rajesh', 'rakesh', 'ram', 'rohan', 'sachin', 'sandeep', 'shubham', 'sumit',
    'sunil', 'suresh', 'tarun', 'varun', 'vijay', 'vikas', 'vinay', 'vivek', 'yash'
}

female_names = {
    'aarti', 'aditi', 'akanksha', 'amrita', 'ananya', 'anjali', 'archana', 'asha',
    'bhavna', 'deepa', 'divya', 'gauri', 'heena', 'isha', 'jyoti', 'kajal', 'kavita',
    'khushi', 'komal', 'kritika', 'lakshmi', 'meena', 'monika', 'muskan', 'neha',
    'nidhi', 'nikita', 'pooja', 'pragya', 'preeti', 'priya', 'rani', 'rekha', 'riya',
    'sakshi', 'sangeeta', 'shalini', 'shreya', 'smita', 'sonal', 'sunita', 'swati',
    'tanvi', 'vidya'
}


In [ ]:
# 4) Utility functions
VALID_EXTS = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff', '.webp'}


def normalize_spaces(text: str) -> str:
    return re.sub(r'\s+', ' ', text).strip()


def clean_name_prefixes(line: str) -> str:
    line = normalize_spaces(line)
    line = re.sub(r'^(Name|Student Name|Candidate Name|S\/O|D\/O)\s*[:\-]\s*', '', line, flags=re.IGNORECASE)
    return normalize_spaces(line)


def to_name_case_if_upper(line: str) -> str:
    if line.isupper():
        return ' '.join(w.capitalize() for w in line.split())
    return line


def is_probable_name_line(line: str) -> bool:
    """
    Name detection rules:
    - 2 to 4 words
    - ignore any digits/special characters (letters, apostrophe, hyphen only)
    - ignore lines longer than 5 words
    """
    line = to_name_case_if_upper(clean_name_prefixes(line))
    if not line:
        return False

    words = line.split(' ')
    if len(words) > 5:
        return False
    if not (2 <= len(words) <= 4):
        return False

    for w in words:
        if not re.fullmatch(r"[A-Za-z]+(?:[-'][A-Za-z]+)?", w):
            return False
        if not w[0].isupper():
            return False
    return True


def parse_indian_name(full_name: str):
    """
    Indian name parsing rules:
    - 1 word  -> First
    - 2 words -> First Last
    - 3+ words -> First Middle Last (compound middle preserved)
    """
    words = normalize_spaces(full_name).split(' ')
    if len(words) == 1:
        return words[0], '', ''
    if len(words) == 2:
        return words[0], '', words[1]

    first = words[0]
    last = words[-1]
    middle = ' '.join(words[1:-1])
    return first, middle, last


def guess_gender(first_name: str) -> str:
    key = first_name.strip().lower()
    if key in male_names:
        return 'Male'
    if key in female_names:
        return 'Female'
    return 'Unknown'


def extract_dob(text: str) -> str:
    """Extract first DOB-like date if present; otherwise blank."""
    patterns = [
        r'\b(\d{2}[/-]\d{2}[/-]\d{4})\b',
        r'\b(\d{4}[/-]\d{2}[/-]\d{2})\b',
        r'\b(\d{2}[/-]\d{2}[/-]\d{2})\b',
    ]
    for pat in patterns:
        match = re.search(pat, text)
        if match:
            return match.group(1)
    return ''


def ocr_text_lines(image_path: str, reader) -> List[str]:
    image = cv2.imread(image_path)
    if image is None:
        return []
    results = reader.readtext(image, detail=0, paragraph=False)
    return [normalize_spaces(t) for t in results if normalize_spaces(t)]


def get_uploaded_image_paths() -> List[str]:
    """
    Colab: interactive upload via files.upload().
    Localhost Jupyter: prompt for image paths separated by commas.
    """
    if IS_COLAB:
        uploaded = colab_files.upload()
        if not uploaded:
            raise ValueError('No images uploaded. Please upload at least one image file.')
        image_paths = [name for name in uploaded.keys() if Path(name).suffix.lower() in VALID_EXTS]
    else:
        raw = input('Enter image path(s), separated by commas: ').strip()
        image_paths = [p.strip() for p in raw.split(',') if p.strip()]
        image_paths = [p for p in image_paths if Path(p).suffix.lower() in VALID_EXTS and Path(p).exists()]

    if not image_paths:
        raise ValueError('No valid image files found. Please provide PNG/JPG/JPEG/BMP/TIF/TIFF/WEBP images.')

    print(f'Loaded {len(image_paths)} file(s):')
    for p in image_paths:
        print('-', p)
    return image_paths


def trigger_download(file_path: str):
    if IS_COLAB:
        colab_files.download(file_path)
    else:
        print(f'Localhost mode: file saved at {Path(file_path).resolve()}')


In [ ]:
# 5) Upload / provide one or multiple images
image_paths = get_uploaded_image_paths()


In [ ]:
# 6) Initialize EasyOCR (English model)
reader = easyocr.Reader(['en'], gpu=False)


In [ ]:
# 7) Process all images and extract student records
records: List[Dict[str, str]] = []

for img_path in image_paths:
    lines = ocr_text_lines(img_path, reader)
    full_text = ' '.join(lines)
    dob_value = extract_dob(full_text)

    seen_names = set()
    for raw_line in lines:
        line = to_name_case_if_upper(clean_name_prefixes(raw_line))
        if is_probable_name_line(line):
            key = line.lower()
            if key in seen_names:
                continue
            seen_names.add(key)

            first, middle, last = parse_indian_name(line)
            gender = guess_gender(first)

            records.append({
                'First Name': first,
                'Middle Name': middle,
                'Last Name': last,
                'DOB': dob_value,
                'Gender': gender,
                'Source Image': Path(img_path).name,
            })

df = pd.DataFrame(records, columns=[
    'First Name', 'Middle Name', 'Last Name', 'DOB', 'Gender', 'Source Image'
])

print(f'Extracted {len(df)} record(s).')


In [ ]:
# 8) Preview extracted table
if df.empty:
    print('No probable names detected from uploaded image(s).')
else:
    display(df)


In [ ]:
# 9) Export to Excel
output_file = 'extracted_students.xlsx'
df.to_excel(output_file, index=False)
print(f'Excel file saved as: {output_file}')


In [ ]:
# 10) Automatic download (Colab) / local file path (localhost)
trigger_download('extracted_students.xlsx')
